## Imports

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy import sparse

RANDOM_STATE = 42

##  Project root

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_dataset.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ml_ready"
MODEL_DIR = PROJECT_ROOT / "models" / "preprocessors"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input:", DATA_PATH)

Project root: /home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler/ml-service/preprocessing
Input: /home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler/ml-service/preprocessing/data/processed/cleaned_dataset.csv


## Load cleaned dataset

In [8]:
df = pd.read_csv("/home/tonoy-sen/Desktop/pro/MlProjects/multidisease-risk-profiler/data/processed/cleaned_dataset.csv")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype("string").str.strip()

print("Shape:", df.shape)
display(df.head())

/tmp/ipykernel_37198/915356809.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


Shape: (278701, 39)


,composite_key,age_level,gender,bmi_level,smoking,diabetes,age,age_normalized,bmi,hypertension,...,salt_intake,heart_rate,hdl,ldl,education_level,employment_status,source_dataset,disease_flags,sublabel,label
0,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,9,0.080460,19.20,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
1,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,22.55,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
2,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,22.89,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
3,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,3,0.011494,23.12,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal
4,Young_Female_Normal_Never_No,Young,Female,Normal,Never,No,4,0.022989,19.61,0.0,...,8.685304,74.329073,65.651757,129.220447,Primary,Retired,diabetes,"0,0,0",N,Normal


## Create targets from confirmed disease_flags mapping.
### disease_flags = Diabetes, Hypertension, Heart Disease

In [9]:
def parse_flags(value):
    if pd.isna(value):
        return (np.nan, np.nan, np.nan)
    bits = str(value).strip().replace(" ", "")
    if bits not in {
        "0,0,0", "0,0,1", "0,1,0", "0,1,1",
        "1,0,0", "1,0,1", "1,1,0", "1,1,1"
    }:
        return (np.nan, np.nan, np.nan)
    return tuple(map(int, bits.split(",")))

flags = df["disease_flags"].apply(parse_flags)
df["diabetes_target"] = flags.apply(lambda x: x[0])
df["hypertension_target"] = flags.apply(lambda x: x[1])
df["heart_disease_target"] = flags.apply(lambda x: x[2])
df["obesity_target"] = df["bmi_level"].astype("string").str.strip()

target_cols = [
    "diabetes_target",
    "hypertension_target",
    "heart_disease_target",
    "obesity_target",
]

## Validate targets

In [10]:
for col in target_cols:
    print("\n", col)
    print(df[col].value_counts(dropna=False).sort_index())

# Diabetes consistency check
if "diabetes" in df.columns:
    original = df["diabetes"].map({"Yes": 1, "No": 0})
    print("\nDiabetes mismatches:",
          (original != df["diabetes_target"]).sum())

# Remove rows missing any of the four targets
df_model = df.dropna(subset=target_cols).copy()
print("\nRows before:", len(df))
print("Rows after :", len(df_model))



 diabetes_target
diabetes_target
0    178083
1    100618
Name: count, dtype: int64

 hypertension_target
hypertension_target
0    272810
1      5891
Name: count, dtype: int64

 heart_disease_target
heart_disease_target
0    140532
1    138169
Name: count, dtype: int64

 obesity_target
obesity_target
Normal         70591
Obese          98377
Overweight     76897
Underweight    32836
Name: count, dtype: Int64

Diabetes mismatches: 0

Rows before: 278701
Rows after : 278701


## Remove metadata, leakage, and redundant columns

In [12]:
GLOBAL_EXCLUDE = {
    "composite_key",
    "source_dataset",
    "disease_flags",
    "sublabel",
    "label",
    "diabetes",
    "hypertension",
    "heart_disease",
    "age_normalized",
    "bmi_level",
    "diabetes_target",
    "hypertension_target",
    "heart_disease_target",
    "obesity_target",
}

common_features = [
    c for c in df_model.columns if c not in GLOBAL_EXCLUDE
]

FEATURES = {
    "diabetes": common_features.copy(),
    "hypertension": common_features.copy(),
    "heart_disease": common_features.copy(),
    "obesity": [c for c in common_features if c != "bmi"],
}

for task, cols in FEATURES.items():
    print(f"\n{task.upper()}: {len(cols)} features")
    print(cols)
    print("Leakage overlap:", GLOBAL_EXCLUDE.intersection(cols))




DIABETES: 29 features
['age_level', 'gender', 'smoking', 'age', 'bmi', 'hba1c_level', 'glucose', 'cholesterol', 'sleep_hours', 'triglycerides', 'physical_activity', 'family_history', 'stress_level', 'low_hdl_cholesterol', 'high_ldl_cholesterol', 'blood_pressure', 'high_blood_pressure', 'sugar_consumption', 'crp_level', 'homocysteine_level', 'systolic_bp', 'diastolic_bp', 'alcohol_intake', 'salt_intake', 'heart_rate', 'hdl', 'ldl', 'education_level', 'employment_status']
Leakage overlap: set()

HYPERTENSION: 29 features
['age_level', 'gender', 'smoking', 'age', 'bmi', 'hba1c_level', 'glucose', 'cholesterol', 'sleep_hours', 'triglycerides', 'physical_activity', 'family_history', 'stress_level', 'low_hdl_cholesterol', 'high_ldl_cholesterol', 'blood_pressure', 'high_blood_pressure', 'sugar_consumption', 'crp_level', 'homocysteine_level', 'systolic_bp', 'diastolic_bp', 'alcohol_intake', 'salt_intake', 'heart_rate', 'hdl', 'ldl', 'education_level', 'employment_status']
Leakage overlap: set(

## Split function: 70/15/15 with stratification

In [13]:
def split_dataset(data, features, target):
    X = data[features].copy()
    y = data[target].copy()

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=y_temp,
    )

    return X_train, X_val, X_test, y_train, y_val, y_test


## Preprocessor: fit only on training data

In [14]:
def make_preprocessor(X):
    numeric = X.select_dtypes(include=["number"]).columns.tolist()
    categorical = X.select_dtypes(
        include=["object", "string", "category", "bool"]
    ).columns.tolist()

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ])

    transformers = []
    if numeric:
        transformers.append(("num", num_pipe, numeric))
    if categorical:
        transformers.append(("cat", cat_pipe, categorical))

    return ColumnTransformer(transformers=transformers, remainder="drop"), numeric, categorical
